In [ ]:
import os

# Fixed profile for quick, stable runs across Colab/Kaggle.
os.environ["MODEL_NAME"] = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-Coder-7B-Instruct")
os.environ["REPO_ID"] = os.environ.get("REPO_ID", "Swastikr/polyglot_optima")
os.environ["EPISODES"] = os.environ.get("EPISODES", "8")
os.environ["FINAL_EVAL_EPISODES"] = os.environ.get("FINAL_EVAL_EPISODES", "12")
os.environ["GRPO_PROMPTS"] = os.environ.get("GRPO_PROMPTS", "240")
os.environ["GRPO_STEPS"] = "300"

# Upload toggle: 0 = local/keyless (default), 1 = publish model/results to HF Hub.
os.environ["UPLOAD_TO_HF"] = os.environ.get("UPLOAD_TO_HF", "0")
if os.environ["UPLOAD_TO_HF"] != "1":
    os.environ["DISABLE_HF_UPLOAD"] = "1"
    os.environ.pop("HF_TOKEN", None)

# W&B toggle: 0 = disabled (default), 1 = enabled (requires WANDB_API_KEY).
os.environ["USE_WANDB"] = os.environ.get("USE_WANDB", "0")
os.environ["WANDB_API_KEY"] = os.environ.get(
    "WANDB_API_KEY",
    "",
)
if os.environ["USE_WANDB"] != "1":
    os.environ.pop("WANDB_API_KEY", None)
else:
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError(
            "USE_WANDB=1 but WANDB_API_KEY is missing. Set WANDB_API_KEY in this cell/runtime env or switch USE_WANDB to 0."
        )

# Keep notebook logs clean from expected verifier numeric edge-case warnings.
# Also suppress noisy regex escape SyntaxWarnings like: invalid escape sequence '\\W'.
os.environ["PYTHONWARNINGS"] = os.environ.get(
    "PYTHONWARNINGS",
    "ignore:invalid value encountered in floor_divide:RuntimeWarning,"
    "ignore:invalid escape sequence:SyntaxWarning",
)

print({
    "MODEL_NAME": os.environ["MODEL_NAME"],
    "REPO_ID": os.environ["REPO_ID"],
    "EPISODES": os.environ["EPISODES"],
    "FINAL_EVAL_EPISODES": os.environ["FINAL_EVAL_EPISODES"],
    "GRPO_PROMPTS": os.environ["GRPO_PROMPTS"],
    "GRPO_STEPS": os.environ["GRPO_STEPS"],
    "UPLOAD_TO_HF": os.environ["UPLOAD_TO_HF"],
    "DISABLE_HF_UPLOAD": os.environ.get("DISABLE_HF_UPLOAD", "0"),
    "USE_WANDB": os.environ["USE_WANDB"],
    "WANDB_API_KEY_SET": bool(os.environ.get("WANDB_API_KEY")),
    "PYTHONWARNINGS": os.environ["PYTHONWARNINGS"],
})


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
IN_NOTEBOOK_RUNTIME = IN_COLAB or IN_KAGGLE

if IN_NOTEBOOK_RUNTIME:
    deps = [
        "openenv>=0.1.13",
        "trl>=0.14.0",
        "transformers>=4.40",
        "datasets>=2.18",
        "peft>=0.11",
        "accelerate>=0.30",
        "huggingface_hub>=0.22",
        "matplotlib>=3.8",
        "wandb>=0.16",
        "bitsandbytes>=0.43.1",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *deps])

    required = {
        "trl": "trl>=0.14.0",
        "transformers": "transformers>=4.40",
        "datasets": "datasets>=2.18",
        "peft": "peft>=0.11",
        "accelerate": "accelerate>=0.30",
        "huggingface_hub": "huggingface_hub>=0.22",
        "openenv": "openenv>=0.1.13",
    }
    for mod, pkg in required.items():
        try:
            importlib.import_module(mod)
        except Exception:
            print(f"[runtime] missing {mod}; installing {pkg}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])

    import trl  # noqa: F401
    print("[runtime] trl version:", trl.__version__)

if IN_COLAB:
    base_root = Path("/content")
elif IN_KAGGLE:
    base_root = Path("/kaggle/working")
else:
    base_root = Path.cwd()

repo_root = base_root / "Openenv-Hack-finale"
if not repo_root.exists():
    subprocess.check_call([
        "git",
        "clone",
        "https://github.com/QuantumByte-01/Openenv-Hack-finale.git",
        str(repo_root),
    ])
else:
    subprocess.check_call(["git", "-C", str(repo_root), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(repo_root), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(repo_root), "reset", "--hard", "origin/main"])

training_dir = repo_root / "polyglot_optima" / "training"
os.chdir(training_dir)
print("Runtime:", "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local")
print("CWD:", Path.cwd())


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

help_proc = subprocess.run(
    [sys.executable, "hf_partial_train.py", "--help"],
    capture_output=True,
    text=True,
)
if help_proc.returncode != 0:
    print("[colab] Warning: hf_partial_train.py --help failed; showing raw error below")
    print("[colab] help stdout:\n", help_proc.stdout)
    print("[colab] help stderr:\n", help_proc.stderr)

    if "No module named 'trl'" in (help_proc.stderr or ""):
        print("[colab] Auto-fixing missing 'trl' and core deps...", flush=True)
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "trl>=0.14.0",
            "transformers>=4.40",
            "datasets>=2.18",
            "peft>=0.11",
            "accelerate>=0.30",
            "bitsandbytes>=0.43.1",
        ])
        help_proc = subprocess.run(
            [sys.executable, "hf_partial_train.py", "--help"],
            capture_output=True,
            text=True,
        )
        print("[colab] Recheck --help return code:", help_proc.returncode)

    print("[colab] Proceeding with conservative fallback flags only.")
    help_text = help_proc.stdout if help_proc.returncode == 0 else ""
else:
    help_text = help_proc.stdout

supports = lambda flag: (flag in help_text)

cmd = [
    sys.executable,
    "hf_partial_train.py",
    "--repo-id", os.environ["REPO_ID"],
    "--model-name", os.environ["MODEL_NAME"],
    "--episodes", os.environ["EPISODES"],
    "--grpo-prompts", os.environ["GRPO_PROMPTS"],
    "--grpo-steps", os.environ["GRPO_STEPS"],
    "--load-in-4bit",
]

# W&B preflight + backward-compat patch for older script versions.
if os.environ.get("USE_WANDB") == "1":
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("USE_WANDB=1 but WANDB_API_KEY is missing")

    os.environ["WANDB_PROJECT"] = os.environ.get("WANDB_PROJECT", "polyglot-optima")
    os.environ["WANDB_MODE"] = "online"

    try:
        import wandb

        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        _probe = wandb.init(
            project=os.environ["WANDB_PROJECT"],
            name=os.environ.get("WANDB_NAME", "colab-grpo-preflight"),
            reinit=True,
        )
        print("[wandb] connected:", _probe.url)
        _probe.finish()
    except Exception as exc:
        print(f"[wandb] preflight warning: {exc}")

    script_path = Path("hf_partial_train.py")
    script_text = script_path.read_text(encoding="utf-8")
    if "report_to_targets" not in script_text and '"report_to": []' in script_text:
        script_text = script_text.replace('"report_to": []', '"report_to": ["wandb"]', 1)
        script_path.write_text(script_text, encoding="utf-8")
        print("[colab] patched hf_partial_train.py to enable report_to=['wandb']")

if supports("--final-eval-episodes"):
    cmd.extend(["--final-eval-episodes", os.environ["FINAL_EVAL_EPISODES"]])
else:
    print("[notebook] --final-eval-episodes not supported in this script version; skipping")

# Legacy-script compatibility: disable SFT/reward-align stages if those flags exist.
if supports("--skip-sft"):
    cmd.append("--skip-sft")
if supports("--reward-align-steps"):
    cmd.extend(["--reward-align-steps", "0"])

# Reduce notebook fragility when GRPO library/version mismatch occurs.
if supports("--allow-grpo-fail"):
    cmd.append("--allow-grpo-fail")

# Optional Hub publishing path (needs HF_TOKEN in environment).
if os.environ.get("UPLOAD_TO_HF") == "1":
    if supports("--publish-model"):
        cmd.extend([
            "--publish-model",
            "--model-repo-id", "Swastikr/polyglot-optima-500step",
            "--public-model",
        ])
    else:
        print("[notebook] publish flags not supported in this script version; skipping")
else:
    if supports("--disable-hf-upload"):
        cmd.append("--disable-hf-upload")

# Verify GPU is active before launching training process.
gpu_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; print(int(torch.cuda.is_available())); print(torch.cuda.device_count()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')",
    ],
    capture_output=True,
    text=True,
)
if gpu_probe.returncode != 0:
    print("[runtime] GPU probe failed:")
    print(gpu_probe.stdout)
    print(gpu_probe.stderr)
    raise RuntimeError("GPU probe failed; restart runtime and enable GPU accelerator")

probe_lines = [x.strip() for x in gpu_probe.stdout.splitlines() if x.strip()]
cuda_ok = (len(probe_lines) >= 1 and probe_lines[0] == "1")
if not cuda_ok:
    raise RuntimeError(
        "No GPU detected. In Colab/Kaggle, enable a GPU accelerator (T4/L4/A100/H100) and rerun."
    )
print(f"[runtime] GPU ready: {probe_lines[-1]}")

print("Running:", " ".join(cmd), flush=True)
start = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
log_tail = []
assert proc.stdout is not None
for line in proc.stdout:
    line = line.rstrip("\n")
    print(line, flush=True)
    log_tail.append(line)
    if len(log_tail) > 120:
        log_tail.pop(0)
rc = proc.wait()
if rc != 0:
    elapsed = time.time() - start
    print(f"[notebook] training failed with code={rc} after {elapsed:.1f}s")
    print("[notebook] Last log lines:")
    for ln in log_tail[-40:]:
        print(ln)
    raise RuntimeError("hf_partial_train.py failed; see printed log tail above")

print("[notebook] GRPO training finished.", flush=True)


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

artifact_dir = Path("/tmp/partial-train-artifacts")
summary_path = artifact_dir / "summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))

for p in [
    artifact_dir / "reward_distribution.png",
    artifact_dir / "baseline_vs_trained_metrics.png",
    artifact_dir / "grpo_reward_curve.png",
    artifact_dir / "grpo_component_means.png",
]:
    if p.exists():
        img = plt.imread(p)
        plt.figure(figsize=(11, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(p.name)
        plt.show()
